<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/InstutionalFiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Institutional Conviction Scoring System
=========================================
Answers: "Is institutional conviction increasing?"

Data source: SEC EDGAR 13F filings (primary, fully free)
             Financial Modeling Prep 13F endpoint (fallback)

Coverage:    US stocks — SP400/SP500 universe

Scoring:
  Metric                          Points
  ─────────────────────────────────────
  Funds increased ownership         0-2
  New funds entered                 0-2
  Top funds accumulating            0-2
  More buyers than sellers          0-2
  Ownership at/near 52-week high    0-2
  ─────────────────────────────────────
  Institutional Score             0-10

Key 13F caveats applied throughout:
  - 45-day reporting lag: "current" quarter data
    reflects holdings as of quarter-end, not today
  - Only managers with >$100M AUM file — smaller
    funds are structurally invisible
  - Long positions only — 13F does not disclose
    short positions, so "increasing ownership"
    cannot be confirmed as purely directionally
    bullish without additional context
  - Shares reported in units of 100 in some
    filings — handled in normalization logic below


In [3]:
pip install requests pandas tabulate

In [14]:
import requests
import pandas as pd

import time
import json
import logging
from pathlib import Path
from datetime import datetime, date
from typing import Optional
import random
import requests
import pandas as pd
import numpy as np

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)
random.seed(42)
print("Libraries Installed!")

Libraries Installed!


In [15]:

# =============================================================
# CONFIG
# =============================================================

FMP_API_KEY        = "sRe2PQAk0MIhqSaoVIOWl69hOyriNNGu"   # free tier at financialmodelingprep.com
CACHE_DIR          = Path("institutional_cache")
CACHE_DIR.mkdir(exist_ok=True)

SEC_HEADERS        = {
    "User-Agent": "obeabi@yahoo.com",
    # SEC EDGAR requires a descriptive User-Agent — use your real
    # name/email here. Requests without a meaningful User-Agent
    # are rejected with 403. This is a policy requirement, not
    # an authentication mechanism.
    "Accept": "application/json",
}

REQUEST_DELAY_SEC  = 0.15   # SEC asks for max 10 req/second; 0.15s is conservative
MAX_RETRIES        = 3

In [16]:

# =============================================================
# 1. SEC EDGAR — CIK LOOKUP
# =============================================================

def get_cik_for_ticker(ticker: str) -> Optional[str]:
    """
    Looks up the SEC CIK (Central Index Key) for a given ticker
    using EDGAR's company search endpoint. CIK is required to
    fetch a company's 13F filing history.

    Returns a zero-padded 10-digit CIK string, or None if not found.
    """
    url = f"https://efts.sec.gov/LATEST/search-index?q=%22{ticker}%22&dateRange=custom&startdt=2020-01-01&enddt=2025-12-31&forms=13F-HR"

    # The more reliable CIK lookup endpoint
    company_url = f"https://www.sec.gov/cgi-bin/browse-edgar?company=&CIK={ticker}&type=13F-HR&dateb=&owner=include&count=1&search_text=&action=getcompany&output=atom"

    # Use the EDGAR company search JSON API directly
    search_url = "https://efts.sec.gov/LATEST/search-index"
    params = {
        "q": f'"{ticker}"',
        "forms": "10-K",  # use 10-K to find the company's own CIK
        "dateRange": "custom",
        "startdt": "2020-01-01",
    }

    try:
        # Best approach: EDGAR's company facts endpoint with ticker->CIK mapping
        tickers_url = "https://www.sec.gov/files/company_tickers.json"
        cache_path = CACHE_DIR / "company_tickers.json"

        if cache_path.exists():
            with open(cache_path) as f:
                tickers_data = json.load(f)
        else:
            resp = requests.get(tickers_url, headers=SEC_HEADERS, timeout=15)
            resp.raise_for_status()
            tickers_data = resp.json()
            with open(cache_path, "w") as f:
                json.dump(tickers_data, f)
            time.sleep(REQUEST_DELAY_SEC)

        # tickers_data is {index: {cik_str, ticker, title}}
        for entry in tickers_data.values():
            if entry.get("ticker", "").upper() == ticker.upper():
                cik = str(entry["cik_str"]).zfill(10)
                return cik

        log.warning(f"CIK not found for {ticker} in EDGAR company tickers list")
        return None

    except Exception as e:
        log.error(f"CIK lookup failed for {ticker}: {e}")
        return None


In [17]:

# =============================================================
# 2. SEC EDGAR — 13F FILING FETCH
# =============================================================

def get_13f_filings_edgar(ticker: str, n_quarters: int = 5) -> list[dict]:
    """
    Fetches up to n_quarters of 13F-HR filings for a given company
    from SEC EDGAR's submissions endpoint.

    Returns a list of dicts, each representing one quarter's filing:
      {
        'quarter':        'Q1 2025',
        'period_of_report': '2025-03-31',
        'filed':          '2025-05-12',
        'total_shares':   int,
        'total_value':    int (in thousands of USD),
        'num_holders':    int,
        'holders':        [{manager, shares, value, type}, ...]
      }

    NOTE: The 13F-HR (Holdings Report) is the standard filing.
    13F-HR/A is an amendment — we use the most recent filing
    (amendment if exists, original otherwise) per period.
    """
    cik = get_cik_for_ticker(ticker)
    if cik is None:
        log.warning(f"Cannot fetch 13F data for {ticker} — CIK not found")
        return []

    cache_path = CACHE_DIR / f"{ticker}_13f_edgar.json"
    if cache_path.exists():
        age_days = (datetime.now() - datetime.fromtimestamp(cache_path.stat().st_mtime)).days
        if age_days < 7:  # cache for 7 days since 13F data is quarterly
            log.info(f"Using cached EDGAR data for {ticker}")
            with open(cache_path) as f:
                return json.load(f)

    log.info(f"Fetching EDGAR submissions for {ticker} (CIK: {cik})")

    try:
        sub_url = f"https://data.sec.gov/submissions/CIK{cik}.json"
        resp = requests.get(sub_url, headers=SEC_HEADERS, timeout=15)
        resp.raise_for_status()
        time.sleep(REQUEST_DELAY_SEC)
        submissions = resp.json()

        # Find 13F-HR filings in the recent filings list
        recent = submissions.get("filings", {}).get("recent", {})
        forms       = recent.get("form", [])
        dates       = recent.get("filingDate", [])
        accessions  = recent.get("accessionNumber", [])
        periods     = recent.get("reportDate", [])

        thirteen_f_filings = []
        for form, filed, accno, period in zip(forms, dates, accessions, periods):
            if form in ("13F-HR", "13F-HR/A"):
                thirteen_f_filings.append({
                    "form": form,
                    "filed": filed,
                    "accession": accno.replace("-", ""),
                    "period": period,
                })

        # Deduplicate: keep most recent filing per period (amendment > original)
        by_period = {}
        for f in thirteen_f_filings:
            p = f["period"]
            if p not in by_period or f["form"] == "13F-HR/A":
                by_period[p] = f

        sorted_periods = sorted(by_period.keys(), reverse=True)[:n_quarters]
        log.info(f"Found {len(sorted_periods)} 13F periods for {ticker}")

        # NOTE: Detailed holdings within each 13F filing require fetching
        # the XML document from the filing index — this is the raw EDGAR path.
        # For most institutional conviction use cases, the FMP endpoint
        # (which already parses this XML) is more practical for the holdings
        # detail. EDGAR is used here for the filing metadata/period discovery.

        results = []
        for period in sorted_periods:
            f = by_period[period]
            results.append({
                "quarter": _period_to_quarter(period),
                "period_of_report": period,
                "filed": f["filed"],
                "accession": f["accession"],
                "cik": cik,
                "source": "SEC_EDGAR",
            })

        with open(cache_path, "w") as f:
            json.dump(results, f, indent=2)

        return results

    except Exception as e:
        log.error(f"EDGAR 13F fetch failed for {ticker}: {e}")
        return []


def _period_to_quarter(period_str: str) -> str:
    """Converts '2025-03-31' -> 'Q1 2025'."""
    try:
        d = datetime.strptime(period_str, "%Y-%m-%d")
        q = (d.month - 1) // 3 + 1
        return f"Q{q} {d.year}"
    except Exception:
        return period_str



In [19]:

# =============================================================
# 3. FMP — 13F INSTITUTIONAL HOLDINGS (PRIMARY HOLDINGS DATA)
# =============================================================

def get_institutional_holders_fmp(ticker: str, n_quarters: int = 5) -> list[dict]:
    """
    Fetches institutional holder data from Financial Modeling Prep.
    FMP's 13F endpoint pre-parses the SEC XML and returns structured
    holder data directly — significantly more practical than parsing
    raw EDGAR XML for each filing.

    Returns a list of quarterly snapshots:
      {
        'quarter':      'Q1 2025',
        'date':         '2025-03-31',
        'holders':      [{investor, shares, change, changeInShares,
                          dateReported, ownership}, ...]
      }

    Free tier limitations:
      - 250 API calls/day
      - Some historical quarters may be gated behind paid tiers
    """
    if FMP_API_KEY == "YOUR_FMP_API_KEY_HERE":
        log.warning("FMP API key not set — skipping FMP data source")
        return []

    cache_path = CACHE_DIR / f"{ticker}_13f_fmp.json"
    if cache_path.exists():
        age_days = (datetime.now() - datetime.fromtimestamp(cache_path.stat().st_mtime)).days
        if age_days < 7:
            log.info(f"Using cached FMP data for {ticker}")
            with open(cache_path) as f:
                return json.load(f)

    log.info(f"Fetching FMP institutional holders for {ticker}")

    url = f"https://financialmodelingprep.com/api/v3/institutional-holder/{ticker}"
    params = {"apikey": FMP_API_KEY}

    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.get(url, params=params, timeout=15)
            if resp.status_code == 429:
                log.warning(f"FMP rate limit hit — waiting 60s (attempt {attempt+1})")
                time.sleep(60)
                continue
            resp.raise_for_status()
            raw = resp.json()
            time.sleep(REQUEST_DELAY_SEC)
            break
        except Exception as e:
            log.error(f"FMP request failed for {ticker} (attempt {attempt+1}): {e}")
            if attempt == MAX_RETRIES - 1:
                return []
            time.sleep(2 ** attempt)

    if not raw or isinstance(raw, dict) and "Error" in raw:
        log.warning(f"FMP returned no data for {ticker}")
        return []

    # Group by reporting date → quarterly snapshots
    df = pd.DataFrame(raw)
    if df.empty:
        return []

    if "dateReported" not in df.columns:
        log.warning(f"FMP response for {ticker} missing dateReported column")
        return []

    df["dateReported"] = pd.to_datetime(df["dateReported"])
    df = df.sort_values("dateReported", ascending=False)

    quarterly = []
    for date_val, group in df.groupby("dateReported"):
        quarter = _period_to_quarter(str(date_val.date()))
        quarterly.append({
            "quarter": quarter,
            "date": str(date_val.date()),
            "holders": group.to_dict("records"),
        })

    quarterly = quarterly[:n_quarters]

    with open(cache_path, "w") as f:
        json.dump(quarterly, f, indent=2, default=str)

    return quarterly


In [20]:

# =============================================================
# 4. CORE ANALYSIS — QoQ METRICS
# =============================================================

def compute_qoq_metrics(quarterly_data: list[dict]) -> dict:
    """
    Given at least two quarters of holder data, computes all the
    QoQ institutional metrics needed for the conviction score.

    Parameters
    ----------
    quarterly_data : list of quarterly snapshots, most recent first,
                     each with 'holders' list containing at minimum
                     {investor, shares, changeInShares, ownership}

    Returns
    -------
    dict with all raw metrics and the final Institutional Score (0-10)
    """
    if len(quarterly_data) < 2:
        return {"error": "Need at least 2 quarters to compute QoQ metrics"}

    current_q  = quarterly_data[0]
    previous_q = quarterly_data[1]

    current_holders  = {h["investor"]: h for h in current_q.get("holders", [])}
    previous_holders = {h["investor"]: h for h in previous_q.get("holders", [])}

    current_names  = set(current_holders.keys())
    previous_names = set(previous_holders.keys())

    # ── New and exited positions ──
    new_positions     = current_names - previous_names
    exited_positions  = previous_names - current_names
    continuing        = current_names & previous_names

    # ── Buyers vs sellers among continuing holders ──
    buyers  = []
    sellers = []

    for name in continuing:
        curr_shares = float(current_holders[name].get("shares", 0) or 0)
        prev_shares = float(previous_holders[name].get("shares", 0) or 0)
        change = curr_shares - prev_shares

        if change > 0:
            buyers.append({"investor": name, "shares_added": change,
                           "curr_shares": curr_shares, "prev_shares": prev_shares})
        elif change < 0:
            sellers.append({"investor": name, "shares_removed": abs(change),
                            "curr_shares": curr_shares, "prev_shares": prev_shares})

    buyers.sort(key=lambda x: x["shares_added"], reverse=True)
    sellers.sort(key=lambda x: x["shares_removed"], reverse=True)

    # ── Aggregate totals ──
    total_shares_current  = sum(
        float(h.get("shares", 0) or 0) for h in current_holders.values()
    )
    total_shares_previous = sum(
        float(h.get("shares", 0) or 0) for h in previous_holders.values()
    )

    shares_change_qoq     = total_shares_current - total_shares_previous
    shares_change_pct     = (
        (shares_change_qoq / total_shares_previous * 100)
        if total_shares_previous > 0 else 0.0
    )

    num_holders_current   = len(current_names)
    num_holders_previous  = len(previous_names)
    holders_change_qoq    = num_holders_current - num_holders_previous

    # ── Ownership % (from most recent data if available) ──
    ownership_pct = None
    for h in current_holders.values():
        own = h.get("ownership")
        if own is not None:
            try:
                ownership_pct = float(own)
                break
            except Exception:
                pass

    # ── Top 10 buyers (new + increasing) ──
    top_buyers = []
    for name in new_positions:
        h = current_holders[name]
        top_buyers.append({
            "investor": name,
            "type": "NEW POSITION",
            "shares": float(h.get("shares", 0) or 0),
        })
    for b in buyers[:10]:
        top_buyers.append({
            "investor": b["investor"],
            "type": "INCREASED",
            "shares_added": b["shares_added"],
            "curr_shares": b["curr_shares"],
        })
    top_buyers = sorted(
        top_buyers,
        key=lambda x: x.get("shares", x.get("shares_added", 0)),
        reverse=True
    )[:10]

    return {
        "current_quarter":        current_q.get("quarter", ""),
        "previous_quarter":       previous_q.get("quarter", ""),
        "num_holders_current":    num_holders_current,
        "num_holders_previous":   num_holders_previous,
        "holders_change_qoq":     holders_change_qoq,
        "total_shares_current":   total_shares_current,
        "total_shares_previous":  total_shares_previous,
        "shares_change_qoq":      shares_change_qoq,
        "shares_change_pct":      round(shares_change_pct, 2),
        "ownership_pct":          ownership_pct,
        "new_positions":          sorted(list(new_positions)),
        "exited_positions":       sorted(list(exited_positions)),
        "num_new_positions":      len(new_positions),
        "num_exited_positions":   len(exited_positions),
        "num_buyers":             len(buyers),
        "num_sellers":            len(sellers),
        "top_10_buyers":          top_buyers,
        "top_10_sellers":         sellers[:10],
        "largest_buyer":          buyers[0]  if buyers  else None,
        "largest_seller":         sellers[0] if sellers else None,
    }


In [21]:

# =============================================================
# 5. 52-WEEK OWNERSHIP HIGH CHECK
# =============================================================

def check_52w_ownership_high(quarterly_data: list[dict]) -> dict:
    """
    Checks whether current total institutional shares ownership is
    at or near its 52-week (4-quarter) high — one of the five
    scoring metrics. Uses up to 5 quarters of data.

    Returns pct_of_peak (1.0 = exactly at 52W high) and a
    boolean is_at_high (within 2% of peak).
    """
    if not quarterly_data:
        return {"pct_of_peak": None, "is_at_high": False}

    shares_by_quarter = []
    for q in quarterly_data:
        total = sum(float(h.get("shares", 0) or 0) for h in q.get("holders", []))
        shares_by_quarter.append(total)

    if not shares_by_quarter:
        return {"pct_of_peak": None, "is_at_high": False}

    current = shares_by_quarter[0]
    peak    = max(shares_by_quarter)

    if peak == 0:
        return {"pct_of_peak": None, "is_at_high": False}

    pct_of_peak = current / peak
    is_at_high  = pct_of_peak >= 0.98   # within 2% of all-time high in window

    return {
        "pct_of_peak": round(pct_of_peak, 4),
        "is_at_high": is_at_high,
        "current_shares": current,
        "peak_shares": peak,
    }

In [22]:

# =============================================================
# 6. INSTITUTIONAL CONVICTION SCORE (0-10)
# =============================================================

def compute_institutional_score(metrics: dict, ownership_high: dict) -> dict:
    """
    Computes the 0-10 Institutional Conviction Score from the five
    component metrics, each contributing 0-2 points.

    Metric                          Points  Logic
    ─────────────────────────────────────────────────────────
    Funds increased ownership         0-2   Based on % of
                                           continuing holders
                                           who increased
    New funds entered                 0-2   Based on number of
                                           new positions vs
                                           total holder count
    Top funds accumulating            0-2   Based on whether
                                           the top 3 largest
                                           holders are buyers
    More buyers than sellers          0-2   Buyer/seller ratio
                                           among continuing
                                           holders
    Ownership at 52-week high         0-2   pct_of_peak from
                                           check_52w_ownership_high
    ─────────────────────────────────────────────────────────
    Total                             0-10
    """
    scores = {}

    # ── Component 1: Funds increased ownership (0-2) ──
    num_buyers    = metrics.get("num_buyers", 0)
    num_sellers   = metrics.get("num_sellers", 0)
    num_holders   = metrics.get("num_holders_current", 1)
    total_active  = num_buyers + num_sellers

    if total_active > 0:
        pct_increasing = num_buyers / total_active
        if pct_increasing >= 0.65:
            scores["funds_increased"] = 2
        elif pct_increasing >= 0.50:
            scores["funds_increased"] = 1
        else:
            scores["funds_increased"] = 0
    else:
        scores["funds_increased"] = 0

    # ── Component 2: New funds entered (0-2) ──
    num_new = metrics.get("num_new_positions", 0)
    if num_holders > 0:
        pct_new = num_new / num_holders
        if pct_new >= 0.10:     # ≥10% of current holders are brand new
            scores["new_funds"] = 2
        elif pct_new >= 0.05:   # ≥5% new
            scores["new_funds"] = 1
        else:
            scores["new_funds"] = 0
    else:
        scores["new_funds"] = 0

    # ── Component 3: Top funds accumulating (0-2) ──
    # Check if the largest buyers are also among the largest holders
    # by shares (a top fund adding = more conviction than a small fund adding)
    top_buyers = metrics.get("top_10_buyers", [])
    if len(top_buyers) >= 3:
        # Any top-3 buyer is a new position or large increaser
        top3_types = [b.get("type", "") for b in top_buyers[:3]]
        new_or_large = sum(1 for t in top3_types if t in ("NEW POSITION", "INCREASED"))
        if new_or_large >= 3:
            scores["top_funds_accumulating"] = 2
        elif new_or_large >= 2:
            scores["top_funds_accumulating"] = 1
        else:
            scores["top_funds_accumulating"] = 0
    elif len(top_buyers) >= 1:
        scores["top_funds_accumulating"] = 1
    else:
        scores["top_funds_accumulating"] = 0

    # ── Component 4: More buyers than sellers (0-2) ──
    if total_active > 0:
        ratio = num_buyers / total_active
        if ratio >= 0.70:       # 70%+ of active holders are buying
            scores["buyers_vs_sellers"] = 2
        elif ratio >= 0.55:     # slight majority buying
            scores["buyers_vs_sellers"] = 1
        else:
            scores["buyers_vs_sellers"] = 0
    else:
        scores["buyers_vs_sellers"] = 0

    # ── Component 5: Ownership at 52-week high (0-2) ──
    pct_of_peak = ownership_high.get("pct_of_peak")
    if pct_of_peak is not None:
        if pct_of_peak >= 0.98:     # at or within 2% of 52W high
            scores["ownership_52w_high"] = 2
        elif pct_of_peak >= 0.90:   # within 10% of 52W high
            scores["ownership_52w_high"] = 1
        else:
            scores["ownership_52w_high"] = 0
    else:
        scores["ownership_52w_high"] = 0

    total_score = sum(scores.values())

    # Conviction level label
    if total_score >= 8:
        conviction = "STRONG ACCUMULATION 🟢🟢"
    elif total_score >= 6:
        conviction = "INCREASING 🟢"
    elif total_score >= 4:
        conviction = "NEUTRAL ⚪"
    elif total_score >= 2:
        conviction = "DECREASING 🔴"
    else:
        conviction = "DISTRIBUTION 🔴🔴"

    return {
        "institutional_score":    total_score,
        "conviction":             conviction,
        "score_breakdown":        scores,
        "pct_of_52w_high":        pct_of_peak,
    }


In [23]:
# =============================================================
# 7. MAIN ANALYSIS FUNCTION — PER TICKER
# =============================================================

def analyze_ticker(ticker: str, n_quarters: int = 5) -> dict:
    """
    Full institutional conviction analysis for a single ticker.
    Tries SEC EDGAR first for filing metadata, FMP for holdings
    detail, with graceful fallback.
    """
    log.info(f"Analyzing institutional conviction for {ticker}")

    # Try FMP first for holdings detail (parsed data vs raw XML)
    quarterly_data = get_institutional_holders_fmp(ticker, n_quarters)

    if not quarterly_data:
        log.info(f"FMP returned no data for {ticker} — EDGAR metadata only available")
        # EDGAR provides filing metadata but not pre-parsed holdings
        # In a production version, you'd parse the 13F XML directly here
        edgar_filings = get_13f_filings_edgar(ticker, n_quarters)
        if not edgar_filings:
            return {
                "ticker": ticker,
                "error": "No institutional data available from either EDGAR or FMP",
            }
        return {
            "ticker": ticker,
            "source": "SEC_EDGAR_METADATA_ONLY",
            "filings_found": len(edgar_filings),
            "most_recent_period": edgar_filings[0]["period_of_report"] if edgar_filings else None,
            "note": "Set FMP_API_KEY to get full holder-level detail",
            "edgar_filings": edgar_filings,
        }

    if len(quarterly_data) < 2:
        return {
            "ticker": ticker,
            "error": f"Only {len(quarterly_data)} quarter(s) of data — need 2+ for QoQ analysis",
            "data": quarterly_data,
        }

    metrics        = compute_qoq_metrics(quarterly_data)
    ownership_high = check_52w_ownership_high(quarterly_data)
    score          = compute_institutional_score(metrics, ownership_high)

    return {
        "ticker":               ticker,
        "source":               "FMP",
        "analysis_date":        str(date.today()),
        "current_quarter":      metrics.get("current_quarter"),
        "previous_quarter":     metrics.get("previous_quarter"),

        # ── Holder counts ──
        "num_holders_current":  metrics["num_holders_current"],
        "num_holders_previous": metrics["num_holders_previous"],
        "holders_change_qoq":   metrics["holders_change_qoq"],

        # ── Share ownership ──
        "total_shares_current":  metrics["total_shares_current"],
        "total_shares_previous": metrics["total_shares_previous"],
        "shares_change_qoq":     metrics["shares_change_qoq"],
        "shares_change_pct":     metrics["shares_change_pct"],
        "ownership_pct":         metrics["ownership_pct"],

        # ── Position changes ──
        "new_positions":         metrics["new_positions"],
        "exited_positions":      metrics["exited_positions"],
        "num_new_positions":     metrics["num_new_positions"],
        "num_exited_positions":  metrics["num_exited_positions"],

        # ── Buyers vs sellers ──
        "num_buyers":            metrics["num_buyers"],
        "num_sellers":           metrics["num_sellers"],
        "top_10_buyers":         metrics["top_10_buyers"],
        "top_10_sellers":        metrics["top_10_sellers"],
        "largest_buyer":         metrics["largest_buyer"],
        "largest_seller":        metrics["largest_seller"],

        # ── 52-week high ──
        "ownership_52w_high":    ownership_high,

        # ── CONVICTION SCORE ──
        "institutional_score":   score["institutional_score"],
        "conviction":            score["conviction"],
        "score_breakdown":       score["score_breakdown"],
    }


In [24]:

# =============================================================
# 8. BATCH SCAN — FULL WATCHLIST
# =============================================================

def scan_watchlist(
    tickers: list[str],
    n_quarters: int = 5,
    min_score: int = 0,
) -> pd.DataFrame:
    """
    Runs institutional conviction analysis across a list of tickers
    and returns a ranked DataFrame sorted by Institutional Score.

    Parameters
    ----------
    tickers     : list of US stock tickers (SP400/SP500 universe)
    n_quarters  : how many quarters of 13F data to pull
    min_score   : optional filter — only return stocks scoring above this

    Returns
    -------
    pd.DataFrame sorted by institutional_score descending
    """
    results = []

    for i, ticker in enumerate(tickers, 1):
        log.info(f"[{i}/{len(tickers)}] {ticker}")
        try:
            result = analyze_ticker(ticker, n_quarters)
            results.append(result)
        except Exception as e:
            log.error(f"Failed on {ticker}: {e}")
            results.append({"ticker": ticker, "error": str(e)})

        # Polite delay between tickers — both SEC and FMP have rate limits
        time.sleep(REQUEST_DELAY_SEC * 3)

    # Flatten to DataFrame
    flat = []
    for r in results:
        if "error" in r and "institutional_score" not in r:
            flat.append({
                "Ticker": r.get("ticker", ""),
                "Error": r.get("error", ""),
                "Score": None,
            })
            continue

        flat.append({
            "Ticker":              r.get("ticker", ""),
            "Score":               r.get("institutional_score"),
            "Conviction":          r.get("conviction", ""),
            "Quarter":             r.get("current_quarter", ""),
            "Holders_Now":         r.get("num_holders_current"),
            "Holders_Prev":        r.get("num_holders_previous"),
            "Holders_Change":      r.get("holders_change_qoq"),
            "Shares_Now":          r.get("total_shares_current"),
            "Shares_Change_Pct":   r.get("shares_change_pct"),
            "New_Positions":       r.get("num_new_positions"),
            "Exited_Positions":    r.get("num_exited_positions"),
            "Buyers":              r.get("num_buyers"),
            "Sellers":             r.get("num_sellers"),
            "Ownership_Pct":       r.get("ownership_pct"),
            "Pct_52W_High":        r.get("ownership_52w_high", {}).get("pct_of_peak"),
            "Score_FundsIncreased":    r.get("score_breakdown", {}).get("funds_increased"),
            "Score_NewFunds":          r.get("score_breakdown", {}).get("new_funds"),
            "Score_TopFunds":          r.get("score_breakdown", {}).get("top_funds_accumulating"),
            "Score_BuyersVsSellers":   r.get("score_breakdown", {}).get("buyers_vs_sellers"),
            "Score_52WH":              r.get("score_breakdown", {}).get("ownership_52w_high"),
        })

    df = pd.DataFrame(flat)

    if "Score" in df.columns:
        df = df.dropna(subset=["Score"])
        df = df[df["Score"] >= min_score]
        df = df.sort_values("Score", ascending=False).reset_index(drop=True)

    return df


In [25]:

# =============================================================
# DISPLAY HELPERS
# =============================================================

def print_ticker_report(result: dict) -> None:
    """Prints a formatted single-ticker conviction report."""
    if "error" in result:
        print(f"\n{result['ticker']}: ERROR — {result['error']}")
        return

    print(f"""
{'='*60}
INSTITUTIONAL CONVICTION REPORT — {result['ticker']}
{'='*60}
Analysis Date    : {result.get('analysis_date', 'N/A')}
Current Quarter  : {result.get('current_quarter', 'N/A')}
Previous Quarter : {result.get('previous_quarter', 'N/A')}
Data Source      : {result.get('source', 'N/A')}

── HOLDER COUNT ──────────────────────────────
  Current      : {result.get('num_holders_current', 'N/A')}
  Previous     : {result.get('num_holders_previous', 'N/A')}
  QoQ Change   : {result.get('holders_change_qoq', 'N/A'):+} funds

── SHARE OWNERSHIP ───────────────────────────
  Current      : {result.get('total_shares_current', 0):,.0f} shares
  Previous     : {result.get('total_shares_previous', 0):,.0f} shares
  QoQ Change   : {result.get('shares_change_pct', 0):+.2f}%
  Ownership %  : {result.get('ownership_pct', 'N/A')}

── POSITION CHANGES ──────────────────────────
  New Positions    : {result.get('num_new_positions', 0)}
  Exited Positions : {result.get('num_exited_positions', 0)}
  Buyers           : {result.get('num_buyers', 0)}
  Sellers          : {result.get('num_sellers', 0)}

── 52-WEEK OWNERSHIP HIGH ────────────────────
  Pct of Peak  : {result.get('ownership_52w_high', {}).get('pct_of_peak', 'N/A')}
  At 52W High  : {result.get('ownership_52w_high', {}).get('is_at_high', False)}

── TOP 10 BUYERS ─────────────────────────────""")

    for b in result.get("top_10_buyers", [])[:5]:
        inv  = b.get("investor", "Unknown")[:35]
        typ  = b.get("type", "")
        shrs = b.get("shares", b.get("shares_added", 0))
        print(f"  {inv:<36} {typ:<14} {shrs:>15,.0f} shares")

    print(f"""
── TOP 5 SELLERS ─────────────────────────────""")
    for s in result.get("top_10_sellers", [])[:5]:
        inv  = s.get("investor", "Unknown")[:35]
        shrs = s.get("shares_removed", 0)
        print(f"  {inv:<36} REDUCED        {shrs:>15,.0f} shares")

    bd = result.get("score_breakdown", {})
    score = result.get("institutional_score", 0)
    conviction = result.get("conviction", "")

    print(f"""
── INSTITUTIONAL CONVICTION SCORE ────────────
  Funds increased ownership  : {bd.get('funds_increased', 0)}/2
  New funds entered          : {bd.get('new_funds', 0)}/2
  Top funds accumulating     : {bd.get('top_funds_accumulating', 0)}/2
  More buyers than sellers   : {bd.get('buyers_vs_sellers', 0)}/2
  Ownership at 52-week high  : {bd.get('ownership_52w_high', 0)}/2
  ─────────────────────────────────────────
  TOTAL SCORE                : {score}/10
  CONVICTION                 : {conviction}

NOTE: 13F data has a ~45-day reporting lag. Holdings
reflect quarter-end positions, not today's posture.
{'='*60}""")

In [28]:

# =============================================================
# EXAMPLE USAGE
# =============================================================

if __name__ == "__main__":

    # ── Single ticker analysis ──
    # Set your FMP API key above before running
    # Free key available at: https://financialmodelingprep.com/register

    #test_ticker = "NVDA"
    #result = analyze_ticker(test_ticker, n_quarters=5)
    #print_ticker_report(result)
    print("Nope!")

    # ── Batch watchlist scan ──
    # Replace with your actual SP400/SP500 watchlist
    # watchlist = ["NVDA", "AAPL", "MSFT", "XLI", "AMD"]
    # df = scan_watchlist(watchlist, n_quarters=5, min_score=4)
    # print(df.to_string())
    # df.to_csv("institutional_conviction_scores.csv", index=False)

    # ── EDGAR-only run (no FMP key needed) ──
    # This gives filing metadata (dates, periods) but NOT holder-level
    # detail. Useful for confirming which quarters have filed and when.
    # edgar_result = get_13f_filings_edgar("NVDA", n_quarters=5)
    # print(json.dumps(edgar_result, indent=2))


Nope!


In [30]:



class InstitutionalEngine:

    BASE_URL = "https://data.businessquant.com/13f"

    def __init__(self, api_key):
        self.api_key = api_key

    def _request(self, mode, ticker):

        url = (
            f"{self.BASE_URL}"
            f"?mode={mode}"
            f"&ticker_issuer={ticker}"
            f"&api_key={self.api_key}"
        )

        r = requests.get(url)

        if r.status_code != 200:
            raise Exception(r.text)

        return r.json()

    def get_summary(self, ticker):

        return self._request("summary", ticker)

    def get_stats(self, ticker):

        return self._request("stats", ticker)

    def get_topholders(self, ticker):

        return self._request("topholders", ticker)

    def build_report(self, ticker):
      # Get summary data - we'll take the most recent quarter
      summary_response = self.get_summary(ticker)
      summaries = summary_response.get("data", [])
      if not summaries:
            raise Exception(f"No data found for ticker {ticker}")

      # Sort by quarter (most recent first) - handles '2025-12-31T00:00:00' format
      latest_summary = sorted(
            summaries,
            key=lambda x: x.get("quarter", ""),
            reverse=True
        )[0]
      # Get top holders
      holders_response = self.get_topholders(ticker)
      holders_data = holders_response.get("data", [])
      #Filter holders to match the latest quarter (if multiple quarters returned)
      latest_quarter = latest_summary["quarter"]
      holders_data = [h for h in holders_data if h.get("quarter") == latest_quarter]
      holders = pd.DataFrame(holders_data)
      # === Build Report ===
      report = {}

      report["Ticker"] = latest_summary.get("ticker")
      report["Quarter"] = latest_summary.get("quarter")
      report["Institution Count"] = latest_summary.get("institutions_total_count")
      report["Bought Count"] = latest_summary.get("institutions_bought_count")
      report["Sold Count"] = latest_summary.get("institutions_sold_count")
      report["Held Count"] = latest_summary.get("institutions_held_count")
      report["Total Shares"] = latest_summary.get("institutions_total_shares")
      report["Ownership %"] = latest_summary.get("institutions_shares_pct_outstanding")
      report["QoQ Shares Change"] = latest_summary.get("shares_changed_qoq")
      report["QoQ %"] = latest_summary.get("shares_changed_qoq_pct")
      report["YoY Shares Change"] = latest_summary.get("shares_changed_yoy")

      report["Top10 Concentration"] = (
            holders["institution_pct"].head(10).sum()
            if not holders.empty else 0
        )

      report["Net Buying"] = (
            latest_summary.get("institutions_bought_shares", 0) -
            latest_summary.get("institutions_sold_shares", 0)
        )

      report["Buy/Sell Ratio"] = (
            latest_summary.get("institutions_bought_count", 0) /
            max(latest_summary.get("institutions_sold_count", 0), 1)
        )

      # Top Buyers & Sellers
      if not holders.empty:
        buyers = holders.sort_values("shares_change_qoq", ascending=False)
        sellers = holders.sort_values("shares_change_qoq")
        report["Top Buyers"] = buyers[[
                "name_filer_short", "shares_change_qoq", "shares_change_qoq_pct"
            ]].head(10).reset_index(drop=True)

        report["Top Sellers"] = sellers[[
                "name_filer_short", "shares_change_qoq", "shares_change_qoq_pct"
            ]].head(10).reset_index(drop=True)
      else:
        report["Top Buyers"] = pd.DataFrame()
        report["Top Sellers"] = pd.DataFrame()

      return report



# instutional score
def institutional_score(report):

    score = 0

    # Holder Sentiment
    if report["Buy/Sell Ratio"] > 1.5:
        score += 20
    elif report["Buy/Sell Ratio"] > 1.2:
        score += 15
    elif report["Buy/Sell Ratio"] > 1:
        score += 10

    # QoQ Ownership
    if report["QoQ %"] > 5:
        score += 20
    elif report["QoQ %"] > 2:
        score += 15
    elif report["QoQ %"] > 0:
        score += 10

    # Net Buying

    if report["Net Buying"] > 0:
        score += 20

    # Ownership

    if report["Ownership %"] > 70:
        score += 15
    elif report["Ownership %"] > 50:
        score += 10

    # Concentration
    concentration = report.get("Top10 Concentration", 0)
    if concentration == 0:
        score += 0
    elif concentration < 20:
        score += 15
    elif concentration < 40:
        score += 10
    else:
        score += 5

    # Buyers vs Sellers

    if report["Bought Count"] > report["Sold Count"]:
        score += 10

    return score

In [31]:
API_KEY =  "9cc8c875a6c2b773eef673e93ced70d7"

engine = InstitutionalEngine(API_KEY)

#report = engine.build_report("NVDA")

#print(report)
#df = pd.DataFrame([report])
# Drop unwanted columns
#df = df.drop(columns=['Top Buyers', 'Top Sellers'], errors='ignore')

#df['Institutional Score'] = df.apply(lambda row: institutional_score(row.to_dict()), axis=1)


#print()


,Ticker,Quarter,Institution Count,Bought Count,Sold Count,Held Count,Total Shares,Ownership %,QoQ Shares Change,QoQ %,YoY Shares Change,Top10 Concentration,Net Buying,Buy/Sell Ratio,Institutional Score
0,NVDA,2025-12-31T00:00:00,5605,2920,2375,310,1.625949e+10,66.411336,218362216.0,1.361265,-501736048.0,0,218362216.0,1.229474,65


In [33]:
def build_institutional_dataframe(tickers, api_key, max_retries=2):
    """
    Process multiple tickers and return a single DataFrame with scores.
    """
    engine = InstitutionalEngine(api_key)
    reports = []

    for ticker in tickers:
        try:
            print(f"Processing {ticker}...", end=" ")

            report = engine.build_report(ticker)
            score = institutional_score(report)

            # Create a flat row for the DataFrame (drop complex columns)
            row = {
                "Ticker": report["Ticker"],
                "Quarter": report["Quarter"],
                "Institution Count": report["Institution Count"],
                "Bought Count": report["Bought Count"],
                "Sold Count": report["Sold Count"],
                "Held Count": report["Held Count"],
                "Ownership %": report["Ownership %"],
                "QoQ %": report["QoQ %"],
                "Buy/Sell Ratio": report["Buy/Sell Ratio"],
                "Net Buying": report["Net Buying"],
                #"Top10 Concentration": report["Top10 Concentration"],
                "Institutional Score": score
            }

            reports.append(row)
            print("✓")

        except Exception as e:
            print(f"✗ Error: {e}")
            continue  # Skip failed tickers

    if not reports:
        raise Exception("No data retrieved for any ticker")

    # Create final DataFrame
    df = pd.DataFrame(reports)

    # Optional: Sort by score (highest first)
    df = df.sort_values(by="Institutional Score", ascending=False).reset_index(drop=True)

    # Drop any unwanted columns (you can modify this list)
    columns_to_drop = ['Top Buyers', 'Top Sellers']  # Add more if needed
    df = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors='ignore')

    return df


# ============================
# HOW TO USE IT
# ============================

tickers = ["FFIV", "ALL", "ABNB", "NVDA"]   # Add your list here

api_key = "9cc8c875a6c2b773eef673e93ced70d7"

df = build_institutional_dataframe(tickers, api_key)

# Display results
print(f"\n✅ Processed {len(df)} tickers successfully!")
display(df)                    # Nice display in Jupyter
# df.to_csv("institutional_scores.csv", index=False)   # Uncomment to save

Processing FFIV... ✗ Error: {"detail":"No data found for the given parameters"}
Processing ALL... ✓
Processing ABNB... ✗ Error: {"detail":"No data found for the given parameters"}
Processing NVDA... ✓

✅ Processed 2 tickers successfully!


,Ticker,Quarter,Institution Count,Bought Count,Sold Count,Held Count,Ownership %,QoQ %,Buy/Sell Ratio,Net Buying,Institutional Score
0,ALL,2025-12-31T00:00:00,1649,778,563,308,74.694049,1.961791,1.381883,3828560.0,70
1,NVDA,2025-12-31T00:00:00,5605,2920,2375,310,66.411336,1.361265,1.229474,218362216.0,65
